# 8. Structured Streaming

The code of this module's lessons in [Learn Apache Spark by Doing](https://dataenglab.com/courses/learn-apache-spark-by-doing/), in lesson order, on Apache Spark 4.2.0 in local mode: one machine, no cluster.

Colab runs this notebook in your own Google account: sign in with Google to run the cells, and use *File → Save a copy in Drive* to keep your changes. Nothing runs on DataEngLab and nothing is installed on your computer.

Run the cells from top to bottom. Under a cell, the result the lesson shows is there to compare with yours.

Not available in Colab: the Spark UI (its port is not reachable from a notebook). Where a lesson sends you to it, run that part on your machine; here, `df.explain()` prints the plan.

In [ ]:
# Setup: Java 17+, PySpark 4.2.0 and the webshop data. About a minute the first time.
import os, re, subprocess, sys, urllib.request, zipfile

PYSPARK_VERSION = "4.2.0"
DATA_URL = 'https://github.com/dataeng-lab/learn-apache-spark-by-doing/raw/v2/datasets.zip'

if "google.colab" in sys.modules:
    version = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    major = re.search(r'version "(?:1\.)?(\d+)', version)
    if not major or int(major.group(1)) < 17:       # PySpark 4 does not start on Java 8 or 11
        subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless",
                       shell=True, check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"], check=True)

if not os.path.isdir("datasets"):
    if DATA_URL is None:
        raise RuntimeError("datasets/ is missing: upload the course's datasets folder in the Files panel")
    urllib.request.urlretrieve(DATA_URL, "datasets.zip")
    zipfile.ZipFile("datasets.zip").extractall(".")

print("datasets:", sorted(os.listdir("datasets")))

**Starting point.** The DataFrames and names the earlier modules built. Not part of the lesson text.

In [ ]:
import shutil

# Streaming queries remember what they processed in their checkpoint folder. Start clean,
# so running the notebook a second time reprocesses the events instead of finding nothing new.
for folder in ("ckpt", "warehouse"):
    shutil.rmtree(folder, ignore_errors=True)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("webshop")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

## A streaming DataFrame is still a DataFrame

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/streaming-as-an-unbounded-table/)

In [ ]:
EVENT_SCHEMA = "event_id STRING, order_id INT, event_type STRING, event_ts TIMESTAMP, amount DOUBLE"

batch  = spark.read.schema(EVENT_SCHEMA).json("datasets/order_events")
stream = spark.readStream.schema(EVENT_SCHEMA).json("datasets/order_events")

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
the same files, two readers:
  spark.read       -> isStreaming = False
  spark.readStream -> isStreaming = True

identical schema:
root
 |-- event_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- amount: double (nullable = true)
```

In [ ]:
query = (
    stream.groupBy("event_type").count()
    .writeStream
    .queryName("by_type")
    .outputMode("complete")
    .format("memory")
    .option("checkpointLocation", "ckpt/by_type")
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

## Event time, processing time and windows

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/event-time-windows/)

In [ ]:
from pyspark.sql import functions as F

windowed = (
    stream
    .withWatermark("event_ts", "10 minutes")
    .groupBy(F.window("event_ts", "5 minutes"), F.col("event_type"))
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("events"))
)

## Watermarks and late data

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/watermarks-and-late-data/)

Not run here: one method call out of context; the cell before shows it in a full query.

```python
.withWatermark("event_ts", "10 minutes")
```

## State, checkpoints and restarts

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/state-checkpoints-and-restarts/)

In [ ]:
query = (
    windowed.writeStream
    .outputMode("append")
    .format("memory").queryName("revenue_5min")
    .option("checkpointLocation", "ckpt/revenue_5min")
    .trigger(availableNow=True)
    .start()
)

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
checkpoint directory layout:
  commits/  (14 files)
  offsets/  (14 files)
  sources/  (0 files)
  sources/0/  (12 files)
  state/  (0 files)
  state/0/  (0 files)
```

**Starting point for the next cell.** Not part of the lesson text.

In [ ]:
# In a notebook, wait for the availableNow run to finish so lastProgress has a batch to report.
query.awaitTermination()

In [ ]:
print(query.lastProgress)   # rows in, processing rate, state rows, batch duration
print(query.status)         # what it is doing right now
query.stop()                # stop cleanly at a batch boundary

## Stream the order events end to end

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/stream-order-events-end-to-end/)

**Starting point for the next cell.** Not part of the lesson text.

In [ ]:
input_dir = "datasets/order_events"
output_dir = "warehouse/revenue_5min"
checkpoint_dir = "ckpt/stream_revenue"

The lesson saves this as `stream_revenue.py`.

In [ ]:
EVENT_SCHEMA = "event_id STRING, order_id INT, event_type STRING, event_ts TIMESTAMP, amount DOUBLE"

events = (
    spark.readStream
    .schema(EVENT_SCHEMA)
    .option("maxFilesPerTrigger", 1)
    .json(input_dir)
)

revenue = (
    events
    .withWatermark("event_ts", "10 minutes")
    .groupBy(F.window("event_ts", "5 minutes"), F.col("event_type"))
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("events"))
)

query = (
    revenue.writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", output_dir)
    .option("checkpointLocation", checkpoint_dir)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

In [ ]:
query.stop()
# ... start the same query again with the same checkpointLocation ...